# 第22章 模型选择与超参数

这个 notebook 用一个小型 Gaia 风格颜色-温度数据集演示：超参数如何控制模型复杂度，为什么不能直接用测试集挑模型，以及为什么交叉验证比单次验证切分更稳。

## 学习目标

- 区分参数和超参数
- 比较单次验证切分和交叉验证
- 看到训练误差下降不等于泛化变好
- 走通最终重训练和独立测试
- 理解为什么测试集不能参与选择

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from pathlib import Path

DATA_PATH = Path('../../data/small/stellar_teff_model_selection_demo.csv').resolve()

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            'source_id': raw['source_id'],
            'bp_rp': float(raw['bp_rp']),
            'teff_k': float(raw['teff_k']),
        })

print(f'Loaded {len(rows)} stellar samples from {DATA_PATH.name}')
print('bp_rp range:', min(row['bp_rp'] for row in rows), 'to', max(row['bp_rp'] for row in rows))
print('teff range:', min(row['teff_k'] for row in rows), 'to', max(row['teff_k'] for row in rows))
print('Python version:', platform.python_version())


## 一个清楚的数据职责划分

这里我们显式保留训练集、验证集和测试集的分工，并另外准备训练+验证集合用于最终重训练。

In [ ]:
validation_ids = {'S004', 'S009', 'S014'}
test_ids = {'S005', 'S010', 'S015'}

train_rows = [row for row in rows if row['source_id'] not in validation_ids | test_ids]
validation_rows = [row for row in rows if row['source_id'] in validation_ids]
train_validation_rows = [row for row in rows if row['source_id'] not in test_ids]
test_rows = [row for row in rows if row['source_id'] in test_ids]

print('train/validation/test sizes:', len(train_rows), len(validation_rows), len(test_rows))
print('validation ids:', sorted(validation_ids))
print('test ids:', sorted(test_ids))


## 一个最小多项式回归工具箱

为了保证 notebook 在最小环境中可运行，我们继续使用手写多项式拟合。这里的超参数就是多项式阶数。

In [ ]:
def solve_linear_system(matrix, vector):
    size = len(vector)
    augmented = [row[:] + [value] for row, value in zip(matrix, vector)]
    for col in range(size):
        pivot_row = max(range(col, size), key=lambda row_index: abs(augmented[row_index][col]))
        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot_value = augmented[col][col]
        if abs(pivot_value) < 1e-12:
            raise ValueError('Singular matrix encountered while fitting polynomial')
        for j in range(col, size + 1):
            augmented[col][j] /= pivot_value
        for row_index in range(size):
            if row_index == col:
                continue
            factor = augmented[row_index][col]
            for j in range(col, size + 1):
                augmented[row_index][j] -= factor * augmented[col][j]
    return [augmented[row_index][size] for row_index in range(size)]

def fit_polynomial(data_rows, degree):
    powers = [
        sum((row['bp_rp'] ** power) for row in data_rows)
        for power in range(2 * degree + 1)
    ]
    matrix = []
    vector = []
    for row_index in range(degree + 1):
        matrix.append([powers[row_index + col_index] for col_index in range(degree + 1)])
        vector.append(sum(row['teff_k'] * (row['bp_rp'] ** row_index) for row in data_rows))
    return solve_linear_system(matrix, vector)

def predict_temperature(coefficients, bp_rp):
    return sum(coefficient * (bp_rp ** power) for power, coefficient in enumerate(coefficients))

def rmse(data_rows, coefficients):
    squared_errors = []
    for row in data_rows:
        prediction = predict_temperature(coefficients, row['bp_rp'])
        squared_errors.append((prediction - row['teff_k']) ** 2)
    return math.sqrt(sum(squared_errors) / len(squared_errors))

candidate_degrees = [1, 2, 3, 5]
print('candidate degrees:', candidate_degrees)


## 单次验证切分

先用一次固定验证切分看趋势：训练误差通常会继续下降，但验证误差未必。

In [ ]:
single_split_results = []
for degree in candidate_degrees:
    coefficients = fit_polynomial(train_rows, degree)
    single_split_results.append({
        'degree': degree,
        'train_rmse': rmse(train_rows, coefficients),
        'validation_rmse': rmse(validation_rows, coefficients),
    })

print('Single validation split results:')
for result in single_split_results:
    print(
        '  degree={}: train RMSE={:.2f}, validation RMSE={:.2f}'.format(
            result['degree'], result['train_rmse'], result['validation_rmse']
        )
    )

best_single_split = min(single_split_results, key=lambda item: item['validation_rmse'])
print('Best degree on one validation split:', best_single_split['degree'])


## 3 折交叉验证

现在我们看多次切分的平均表现，估计哪一档复杂度更稳。

In [ ]:
folds = [
    {'S001', 'S006', 'S011', 'S014'},
    {'S002', 'S007', 'S009', 'S013'},
    {'S003', 'S004', 'S008', 'S012'},
]

cv_results = []
for degree in candidate_degrees:
    fold_rmses = []
    for held_out_ids in folds:
        fold_train = [row for row in train_validation_rows if row['source_id'] not in held_out_ids]
        fold_valid = [row for row in train_validation_rows if row['source_id'] in held_out_ids]
        coefficients = fit_polynomial(fold_train, degree)
        fold_rmses.append(rmse(fold_valid, coefficients))
    cv_results.append({
        'degree': degree,
        'fold_rmses': fold_rmses,
        'mean_cv_rmse': sum(fold_rmses) / len(fold_rmses),
    })

print('3-fold cross-validation results:')
for result in cv_results:
    rounded = [round(value, 2) for value in result['fold_rmses']]
    print(
        '  degree={}: fold RMSEs={}, mean CV RMSE={:.2f}'.format(
            result['degree'], rounded, result['mean_cv_rmse']
        )
    )

best_cv = min(cv_results, key=lambda item: (item['mean_cv_rmse'], item['degree']))
print('Best degree from cross-validation:', best_cv['degree'])


## 最终重训练与独立测试

这里才允许测试集出现：先冻结由交叉验证选出的阶数，再用训练+验证数据重训练，最后报告一次测试结果。

In [ ]:
final_degree = best_cv['degree']
final_coefficients = fit_polynomial(train_validation_rows, final_degree)
final_test_rmse = rmse(test_rows, final_coefficients)

print('Final degree chosen after CV:', final_degree)
print('Final test RMSE: {:.2f} K'.format(final_test_rmse))

print('Test-set predictions:')
for row in test_rows:
    prediction = predict_temperature(final_coefficients, row['bp_rp'])
    print('  {}: true={:.0f} K, predicted={:.1f} K'.format(row['source_id'], row['teff_k'], prediction))


## 比较：单次验证切分 vs 交叉验证

这一步帮助我们看到“看起来最好”和“更稳定”之间并不总是同一个选择。

In [ ]:
single_split_degree = best_single_split['degree']
single_split_coefficients = fit_polynomial(train_validation_rows, single_split_degree)
single_split_test_rmse = rmse(test_rows, single_split_coefficients)

print('Comparison of two selection strategies:')
print('  degree chosen by one validation split:', single_split_degree)
print('  degree chosen by cross-validation:', final_degree)
print('  test RMSE after one-split choice: {:.2f} K'.format(single_split_test_rmse))
print('  test RMSE after cross-validation choice: {:.2f} K'.format(final_test_rmse))


## 一个故意的反例：偷看测试集会怎样

下面这个比较只是为了教学示意，不是正确流程。它展示了“如果直接拿测试集挑阶数，会多么诱人”。

In [ ]:
test_peek_results = []
for degree in candidate_degrees:
    coefficients = fit_polynomial(train_validation_rows, degree)
    test_peek_results.append((degree, rmse(test_rows, coefficients)))

best_by_test_peeking = min(test_peek_results, key=lambda item: item[1])
print('Forbidden test-peeking comparison:')
print('  candidate test RMSEs =', [(degree, round(value, 2)) for degree, value in test_peek_results])
print('  best degree if you illegally peeked at test set =', best_by_test_peeking[0])


## Model Selection Artifact Export

这一段导出模型选择日志。重点是保留候选空间、验证设计、最终选择和测试集打开时机，而不是只保留一个最佳阶数。

In [ ]:
RESULTS_DIR = Path('results/part3_evidence_artifacts')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

single_lines = [
    f"- degree={item['degree']}: train_rmse={item['train_rmse']:.2f}, validation_rmse={item['validation_rmse']:.2f}"
    for item in single_split_results
]
cv_lines = [
    f"- degree={item['degree']}: fold_rmses={[round(value, 2) for value in item['fold_rmses']]}, mean_cv_rmse={item['mean_cv_rmse']:.2f}"
    for item in cv_results
]
test_peek_lines = [f"- degree={degree}: test_rmse={value:.2f}" for degree, value in test_peek_results]

artifact = f'''# Ch22 Model Selection Artifact

## Candidate Space

- Model family: polynomial regression from bp_rp to teff_k.
- Hyperparameter search range: degree in {candidate_degrees}.
- Selection rule: choose the lowest mean cross-validation RMSE; break ties toward simpler degree.

## Data Roles

- Training rows: {[row['source_id'] for row in train_rows]}
- Validation rows: {sorted(validation_ids)}
- Test rows: {sorted(test_ids)}
- Test set opened when: after cross-validation selected degree {final_degree} and final coefficients were frozen.

## Single Validation Split

{chr(10).join(single_lines)}

## Cross-Validation

{chr(10).join(cv_lines)}

## Final Test

- Final selected degree: {final_degree}
- Final test RMSE: {final_test_rmse:.2f} K
- One-split selected degree: {single_split_degree}; one-split test RMSE: {single_split_test_rmse:.2f} K

## Leakage / Test-Peeking Demonstration

The following comparison is forbidden for model selection because it uses the test set as if it were validation evidence:

{chr(10).join(test_peek_lines)}

- Best degree if test set is illegally used for selection: {best_by_test_peeking[0]}

## Claim Boundary

This artifact supports a claim about controlled model selection in a tiny teaching regression task. It cannot support an astrophysical calibration of color-temperature relations.
'''

output_path = RESULTS_DIR / 'ch22_model_selection_artifact.md'
output_path.write_text(artifact, encoding='utf-8')
print('wrote', output_path)
print('final degree:', final_degree)


## 小结

这个例子说明：模型选择不是为了在所有分数里挑一个最好看的数字，而是为了用验证过程找到更稳定、可复现的复杂度，再把测试集留到最后做独立证据。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'best_single_split_degree': best_single_split['degree'],
    'best_cv_degree': best_cv['degree'],
    'final_test_rmse_k': round(final_test_rmse, 2),
    'single_split_test_rmse_k': round(single_split_test_rmse, 2),
    'test_peek_degree': best_by_test_peeking[0],
    'python_version': platform.python_version(),
}

print('Model-selection snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
